# 第 29 章 · 评测：冻结分母，分开判断四种结果

Agent 说完成、命令退出零、业务测试通过、真实服务可用，这些结论不能互相代替。评测计划要先冻结 case、attempt、模式和维度，再接收外部 Judge 的记录。

本章只实现聚合器。输入 Judge 记录是显式合成数据，真实写入的证据文件只用于测试摘要绑定；它们不证明任务、沙箱或服务行为。

**运行范围：局部机制实验。** 按顺序执行，每个实验都通过真实函数、文件或队列产生结果，再用断言检查。修改参数后重跑该实验，必要时先重启内核并执行全部。所有合成输入都只证明本地机制；本章没有累计检查点，真实服务与设备仍待验证。

本章使用原作者的设计与局部原型；下面明确写出的简化是教材选择，不是 Codex 原样实现。固定源码：`c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee`。

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter

## 1 · 未运行项仍占分母

先冻结两个 attempt，只提供一个记录。输出应保留另一个 not_run，避免只统计“成功返回的请求”。required_dimensions 规定本 case 的 gate，但四种维度全部保留。

In [2]:
import copy
import hashlib
m = load_example('29', 'aggregate')
run_dir = fresh_run_dir('ch29')
raw_path = run_dir/'synthetic-judge.txt'
raw_path.write_text('synthetic judge input; not a task success record\n')
ref = {'path': raw_path.name, 'sha256': hashlib.sha256(raw_path.read_bytes()).hexdigest()}
def slot(number, required=('task_quality', 'mechanism')):
    return {'slot_id': f'slot-{number}', 'case_id': f'case-{number}', 'attempt_id': f'attempt-{number}',
            'mode': 'fixture', 'target_id': 'local-prototype', 'platform': 'local',
            'required_dimensions': list(required)}
plan = {'schema': 'evaluation-plan-v1', 'slots': [slot(1), slot(2)]}
def record(slot_value, **states):
    dimensions = {d: {'status': states.get(d, 'not_run'), 'reason': 'synthetic unit input',
                     'evidence': [ref] if states.get(d) in {'passed', 'failed'} else []}
                  for d in m.DIMENSIONS}
    return {'schema': 'evaluation-record-v1', 'plan_sha256': m.digest(plan),
            **{k: slot_value[k] for k in ('slot_id', *m.IDENTITY)}, 'dimensions': dimensions}
first = record(plan['slots'][0], task_quality='passed', mechanism='passed')
summary = m.aggregate(plan, [first], run_dir)
assert summary['planned_attempts'] == 2 and summary['recorded_attempts'] == 1
assert summary['dimensions']['task_quality']['verified_pass_fraction'] == 0.5
assert summary['case_gates'] == {'passed': 1, 'not_run': 1}
assert summary['dimensions']['service_runtime']['counts']['not_run'] == 2
print({'case_gates': summary['case_gates'], 'task_quality': summary['dimensions']['task_quality']})

{'case_gates': {'passed': 1, 'not_run': 1}, 'task_quality': {'denominator': 2, 'counts': {'passed': 1, 'failed': 0, 'blocked': 0, 'unknown': 0, 'interrupted': 0, 'not_run': 1}, 'verified_pass_fraction': 0.5}}


## 2 · “能识别错误”的反例也可以通过机制合同

一个故意错误的候选应当 task_quality=failed、mechanism=passed。这并不矛盾：本反例的 gate 只要求检测机制生效。修改 required_dimensions 加入 task_quality，gate 就应失败。

In [3]:
negative_plan = {'schema': 'evaluation-plan-v1', 'slots': [slot(3, ('mechanism',))]}
negative = record(negative_plan['slots'][0], task_quality='failed', mechanism='passed', artifact_identity='passed')
negative['plan_sha256'] = m.digest(negative_plan)
negative_summary = m.aggregate(negative_plan, [negative], run_dir)
assert negative_summary['case_gates'] == {'passed': 1}
assert negative_summary['dimensions']['task_quality']['counts']['failed'] == 1
strict_plan = copy.deepcopy(negative_plan)
strict_plan['slots'][0]['required_dimensions'].append('task_quality')
strict = copy.deepcopy(negative)
strict['plan_sha256'] = m.digest(strict_plan)
assert m.aggregate(strict_plan, [strict], run_dir)['case_gates'] == {'failed': 1}
print({'detection_contract': 'passed', 'synthetic_candidate_quality': 'failed'})

{'detection_contract': 'passed', 'synthetic_candidate_quality': 'failed'}


## 3 · 旧计划、缺证据和改字节都不能借用绿灯

哈希只能证明字节一致，不能证明生产者可信；即便如此，缺哈希校验仍会允许记录指向后来被替换的文件。三个反例分别调用同一入口。

In [4]:
wrong_plan = copy.deepcopy(first)
wrong_plan['plan_sha256'] = 'stale-plan'
missing = copy.deepcopy(first)
missing['dimensions']['task_quality']['evidence'] = []
for candidate, expected in [(wrong_plan, 'wrong_plan'), (missing, 'conclusive_state_without_evidence')]:
    try:
        m.aggregate(plan, [candidate], run_dir)
    except ValueError as error:
        assert str(error) == expected
    else:
        raise AssertionError(expected)
saved_bytes = raw_path.read_bytes()
raw_path.write_text('changed after evidence collection')
try:
    m.aggregate(plan, [first], run_dir)
except ValueError as error:
    assert str(error) == 'evidence_hash_mismatch'
else:
    raise AssertionError('改字节未被检测')
finally:
    raw_path.write_bytes(saved_bytes)
print('旧计划、缺证据、证据被改写均被拒绝')

旧计划、缺证据、证据被改写均被拒绝


## 4 · 中断与重试保留独立 attempt

中断不能从分母消失，也不能被后来的成功覆盖。恢复后新建 attempt，并把原记录重新绑定到新的冻结计划；保留原计划摘要可追溯规划变更。这里没有真实取消进程，只验证评测聚合语义。

In [5]:
interrupted = record(plan['slots'][1], task_quality='interrupted', mechanism='unknown')
partial = m.aggregate(plan, [first, interrupted], run_dir)
assert partial['dimensions']['task_quality']['counts']['interrupted'] == 1
resume_plan = copy.deepcopy(plan)
retry_slot = slot(4)
retry_slot['case_id'] = plan['slots'][1]['case_id']
resume_plan['slots'].append(retry_slot)
records = [copy.deepcopy(first), copy.deepcopy(interrupted), record(retry_slot, task_quality='passed', mechanism='passed')]
for item in records:
    item['plan_sha256'] = m.digest(resume_plan)
resumed = m.aggregate(resume_plan, records, run_dir)
assert resumed['planned_attempts'] == 3
assert resumed['dimensions']['task_quality']['counts']['interrupted'] == 1
assert resumed['dimensions']['task_quality']['verified_pass_fraction'] == 2/3
print({'original_plan': m.digest(plan), 'resumed_plan': m.digest(resume_plan),
       'quality_counts': resumed['dimensions']['task_quality']['counts']})

{'original_plan': '822fa9116b412343690e130aa1fb9d9e57a8fcbe6472230587c5adcff22ffe9a', 'resumed_plan': '9b880980368b7d98ff3f96ad0a3bfff6f2c351461f7a9cae70d3b087183d686a', 'quality_counts': {'passed': 2, 'failed': 0, 'blocked': 0, 'unknown': 0, 'interrupted': 1, 'not_run': 0}}


## 源码对照与接回主线

以下为固定提交的已定位源码入口。阅读源码能解释设计边界，不等于执行过其测试；本章 Python 原型与这些 Rust 接口不是同一实现。

- [codex-rs/core/src/tools/spec_plan.rs#L1482-L1489](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/tools/spec_plan.rs#L1482-L1489)
- [codex-rs/core/src/tools/handlers/test_sync.rs#L141-L184](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/tools/handlers/test_sync.rs#L141-L184)
- [codex-rs/core/tests/suite/tool_parallelism.rs#L79-L113](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/tests/suite/tool_parallelism.rs#L79-L113)

**接回条件：** 将局部机制接入同一 Dispatcher、Scheduler、History 和受保护验证器，再执行原合同。目录、断言和演示输出都不能替代累计集成证据。